# Stackformer Distributed Training: DDP Multi-Process Launch

This tutorial demonstrates how to launch single-node multi-process Distributed Data Parallel (DDP) training using `stackformer.training.launch`.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

from stackformer.config import ModelConfig
from stackformer.models import GPT2
from stackformer.training import Trainer, TrainConfig, launch

## 1. Define the Worker Function

The worker function is executed simultaneously on each spawned process. Inside, `Trainer` detects the distributed process group and wraps the model with DDP automatically.

In [ ]:
def distributed_train_worker():
    config = ModelConfig(
        vocab_size=500,
        embed_dim=64,
        num_layers=2,
        num_heads=2,
        seq_len=32,
        hidden_dim=128,
    )
    model = GPT2(config)

    x = torch.randint(0, 500, (32, 32))
    y = torch.randint(0, 500, (32, 32))
    train_loader = DataLoader(TensorDataset(x, y), batch_size=4)

    cfg = TrainConfig(
        parallel="ddp",
        max_epochs=1,
        lr=1e-3,
        verbosity="normal",
    )
    trainer = Trainer(model, cfg, train_loader=train_loader)
    trainer.train()
    print("Worker finished training successfully!")

## 2. Launch Multi-Process Training

In [ ]:
# Launch 2 CPU/gloo processes
launch(distributed_train_worker, nproc_per_node=2, backend="gloo")